In [22]:
import pandas as pd
import numpy as np

np.random.seed(42)

dataset_path = '../assets/processed/'
X_train, X_dev, X_test, = pd.read_csv(f'{dataset_path}X_train.csv'), pd.read_csv(f'{dataset_path}X_dev.csv'), pd.read_csv(f'{dataset_path}X_test.csv')
Y_train, Y_dev, Y_test, = pd.read_csv(f'{dataset_path}Y_train.csv'), pd.read_csv(f'{dataset_path}Y_dev.csv'), pd.read_csv(f'{dataset_path}Y_test.csv')

X_train, X_dev, X_test = X_train.to_numpy(dtype='float32').T, X_dev.to_numpy(dtype='float32').T, X_test.to_numpy(dtype='float32').T # (n_features, n_samples)
Y_train, Y_dev, Y_test = Y_train.to_numpy(dtype='float32').T, Y_dev.to_numpy(dtype='float32').T, Y_test.to_numpy(dtype='float32').T # (1, n_samples)

### Parameters initialization

In [ ]:
learning_rate = 0.001
batch_size = 128 # Minibatch size
epochs = 100 # 128 * 100 = 12,800 iterations
units = [X_train.shape[0], 128, 64, 32, 1] # input, hidden, output layers (shape[0] = n_features)
activation = 'relu'
lambd = 0.01 # L2 regularization parameter

parameters = {}
for layer in range(1, len(units)):
    # W: (layer units, layer-1 units)
    parameters[f'W{layer}'] = np.random.randn(units[layer], units[layer - 1]) * np.sqrt(2 / units[layer - 1]) # He initialization
    parameters[f'b{layer}'] = np.zeros((units[layer], 1))

batches = []
for i in range(0, X_train.shape[1], batch_size): # iterate over samples (axis 1)
    batches.append((X_train[:, i:i + batch_size], Y_train[:, i:i + batch_size]))

In [24]:
def activation_func(activaton, Z):
    if activaton == 'relu':
        return np.maximum(0, Z)
    if activaton == 'sigmoid':
        return 1 / (1 + np.exp(-Z))

In [25]:
def derivative_activation_func(activaton, Z):
    if activaton == 'relu':
        return np.where(Z > 0, 1, 0) # derivative of ReLU (1 if Z > 0 else 0)

In [ ]:
def loss_func(Y_hat, Y, parameters, lambd):
    m = Y.shape[1]
    cross_entropy_loss = -1/m * np.sum(Y * np.log(Y_hat + 1e-8) + (1 - Y) * np.log(1 - Y_hat + 1e-8)) # epsilon to avoid log(0)
    l2_regularization_loss = (lambd / (2 * m)) * sum(np.sum(np.square(parameters[f'W{layer}'])) for layer in range(1, len(units))) # frobenius norm of weights
    return cross_entropy_loss + l2_regularization_loss

In [ ]:
def forward_propagation(X, L, activation, parameters):
    cache = {}
    A = X # A0 (input layer)

    for layer in range(1, L + 1):
        W = parameters[f'W{layer}']
        b = parameters[f'b{layer}']
        
        Z = np.dot(W, A) + b # Z = W * A + b
        A = activation_func(activation, Z) if layer < L else activation_func('sigmoid', Z) # sigmoid for the output layer
        cache[f'Z{layer}'], cache[f'A{layer}'] = Z, A

    return A, cache

In [ ]:
def backward_propagation(X, dZ, m, L, activation, parameters, cache):
    grads = {}

    for layer in reversed(range(1, L + 1)):
        A_prev = cache[f'A{layer - 1}'] if layer > 1 else X
        W = parameters[f'W{layer}']
        
        grads[f'dW{layer}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{layer}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
        if layer > 1:
            dA_prev = np.dot(W.T, dZ)
            dZ = dA_prev * derivative_activation_func(activation, cache[f'Z{layer - 1}'])

    return grads

### Model training

In [ ]:
for epoch in range(epochs):
    for X_batch, Y_batch in batches:
        m = X_batch.shape[1] # samples in this batch
        L = len(units) - 1

        # forward prop
        Y_hat, cache = forward_propagation(X_batch, L, activation, parameters)

        # loss with regularization
        loss = loss_func(Y_hat, Y_batch, parameters, lambd)

        # back prop
        dZ = Y_hat - Y_batch # sigmoid + BCE chain derivative (last layer)
        grads = backward_propagation(X_batch, dZ, m, L, activation, parameters, cache)

        # Update parameters with L2 regularization
        for layer in range(1, L + 1):
            parameters[f'W{layer}'] -= learning_rate * (grads[f'dW{layer}'] + (lambd / m) * parameters[f'W{layer}'])
            parameters[f'b{layer}'] -= learning_rate * grads[f'db{layer}']

    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss}')